<a href="https://colab.research.google.com/github/debswrik/genai_codebase/blob/main/fine_tuning_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



*   Peft library is for parameter efficient transfer learning.
Inside peft we can find LoRA (low rank adaptation of large language models).
    peft actually uses technique, which will try to freeze most of the weights of the llm model (only some of the weights will be retrained and based on that they will be able to provide accurate results on the custom dataset) when it applies transfer learning on the LLM models.
*   bitsandbytes library is used for quantization




In [1]:
# !pip install -q accelerate==0.21.0 peft==0.4.0 bitsandbytes==0.49.2 transformers==4.41.2 trl==0.4.7
!pip install -q accelerate==0.30.1 peft==0.11.1 bitsandbytes transformers==4.41.2 trl datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 19.1 MB/s eta 0:00:00


In [10]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    Pipeline,
    logging
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer,SFTConfig
os.environ["PYTORCH_CUDE_ALLOC_CONF"] = "expandable_segments:True"

In [11]:
dataset_name = "debsubhra/guanaco-1k"
dataset = load_dataset(dataset_name)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 1000
    })
})


* #### free google colab offers 15gb graphics card (limited resource)
* #### we also need to consider the overhead due to optimizer states, gradients and forward activation
* #### full fine-tuning is not possible, we need parameter-efficient fine-tuning (PEFT) techniques like LoRA or QLoRA
* #### we would be fine tuning the model with 4-bit precision

* ### QLoRA will use a rank of 64 with a scaling parameter of 16. We'll load the Llama 2 model directly in 4-bit precision using the NF4 type and train it for one epoch.
* ### QLORA will use a rank of 64 with a scaling parameter of 16. We are loading the Llama2 model directly in 4-bit precision using the NF4 type and train it for 1 epoch.
  #### Higher rank --> more learning capacity, more memory
  #### lower rank  --> faster, less memory

* ### scaling parameter controls how strongly LoRA updates affect model

* ### 4-bit precision (NF4) -> model weights stored in 4bits instead of 16/32 bits --> massive memory reduction

In [12]:
from google.colab import userdata
hf_token = userdata.get("hf_token")

In [13]:
# The model we want to train from Hugging Face hub
model_name = "NousResearch/Llama-2-7b-chat-hf"

# The instruction dataset to use
dataset_name = "debsubhra/guanaco-1k"

# Fine-tuned model name
new_model = "Llama-2-7b-chat-hf-finetune-test"

###############################################
#QLoRA parameters
###############################################
# LoRA attention dimension --> Rank of LoRA : this is a hyperparameter
# which controls how much knowledge can be learned
# value  | effect
# -------|---------------
# 8-16   | light tuning
# 32-64  | balanced
# 128+   | heavy tuning

lora_r = 64

# Alpha parameter for LoRA scaling --> scaling parameter, controls
# strength of LoRA updates
lora_alpha = 16

# Dropout probability for LoRA layers, prevents overfitting,
# especially important for small dataset
lora_dropout = 0.1

###############################################
# bitsandbytes parameters
###############################################

# activate 4-bit precision base model loading
use_4bit = True

# compute dtype for 4-bit base models
bnb_4bit_quant_dtype = "float16"

# quantization type (nf4 / fp4)
bnb_4bit_quant_type = "nf4"

# activate nested quantization for 4-bit base models (double quantization)
use_nested_quant = False

###############################################
# TrainingArguments parameters
###############################################

# o/p directory where the model predictions and checkpoints will be stored
output_dir = "./results"

# number of training epochs
num_train_epochs = 1

# enable fp16/bf16 training (set bf16 to true with an A100)
fp16 = True
bf16 = False

# Batch size per GPU for training
per_device_train_batch_size = 1

# Number of update steps to accumulate the gradients for
gradient_accumulation_steps = 4

# enable gradient checkpointing
gradient_checkpointing = True

# maximum gradient normal (gradient clipping)
max_grad_norm = 0.3

# Initial learning rate (Adam optimizer)
learning_rate = 2e-4

# Weight decay to apply to all layers except bias/LayerNorm weights
weight_decay = 0.001

# optimizer to use
optim = "paged_adamw_32bit"

# learning rate schedule
lr_scheduler_type = "cosine"

# number of training steps (overrides num_train_epochs)
max_steps = -1

# ratio of steps for a linear warmup (from 0 to learning rate)
warmup_ratio = 0.03

# group sequences into batches with same length
# saves memory and speeds up training considerably
group_by_length = True

# save checkpoint every X updates steps
save_steps = 0

# log every X updates steps
logging_steps = 25


#######################################
# SFT parameters
#######################################

# maximum sequence length to use
max_seq_length = 512

# pack multiple short examples in the same input sequence to increase
# efficiency
packing = False

# Load the entire model on the GPU 0
device_map = {"": 0}



* # Load everything and start the fine tuning
  ### Load the dataset
  ### configure bitsandbytes for 4-bit quantization
  ### Load the Llama 2 model in 4-bit precision on a GPU with the corresponding tokenizor
  ### Load the configuration of QLoRA, regular training parameters and passing everything to the SFTtrainer.

In [15]:
from transformers.utils import quantization_config
dataset_name = "debsubhra/guanaco-1k"
dataset = load_dataset(dataset_name, split="train")

compute_dtype = getattr(torch, bnb_4bit_quant_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

# Check GPU compatibility with bfloat16
if compute_dtype == torch.float16 and use_4bit:
  major,_=torch.cuda.get_device_capability()
  if major >= 8:
    print("=" * 80)
    print("Your GPU supports bfloat16: accelerate training with bf16=True")
    print("=" * 80)


# Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map = device_map
)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.config.use_cache = False
model.config.pretraining_tp = 1

# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.per_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # Fix weird overflow issue with FP16 training

# Load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CASUAL_LM"
)

# Set training parameters
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to="tensorboard"
)

sft_config = SFTConfig(
    max_seq_length=max_seq_length,
    dataset_text_field="text",
    packing=False,
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to="tensorboard"
)
# Set supervised fine-tuned parameters
# trainer=SFTTrainer(
#     model=model,
#     train_dataset=dataset,
#     peft_config=peft_config,
#     dataset_text_field="text",
#     max_seq_length=max_seq_length,
#     tokenizer=tokenizer,
#     args=training_arguments,
#     packing=packing
# )
trainer=SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    tokenizer=tokenizer,
    args=sft_config,
)

trainer.train()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Step,Training Loss
25,1.354900
50,1.636800
75,1.239400
100,1.440800
125,1.227200
150,1.369100
175,1.222500
200,1.455900
225,1.221400
250,1.510300


TrainOutput(global_step=250, training_loss=1.3678349075317382, metrics={'train_runtime': 1643.2932, 'train_samples_per_second': 0.609, 'train_steps_per_second': 0.152, 'total_flos': 1.3981667059482624e+16, 'train_loss': 1.3678349075317382, 'epoch': 1.0})

In [16]:
from transformers import pipeline
prompt = "how can I create a timestamp from a date with the Carbon php package?"

pipe = pipeline(
    task = "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=200
)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


<s>[INST] how can I create a timestamp from a date with the Carbon php package? [/INST]  You can create a timestamp from a date using the `Carbon` package in PHP by using the `now()` method, which returns a `Carbon` instance representing the current date and time.

Here's an example:
```
use Carbon\Carbon;

$date = '2023-03-14'; // input date

$timestamp = Carbon::now(); // create a timestamp from the current date and time

echo $timestamp->format('Y-m-d'); // output the timestamp in the format YYYY-MM-DD
```
In this example, we create a `Carbon` instance representing the current date and time using the `now()` method. Then, we use the `format()` method to output the timestamp in the format `YYYY-


In [18]:
from transformers import pipeline
prompt = "why can joe rogan take over the world?"

pipe = pipeline(
    task = "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=200
)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


<s>[INST] why can joe rogan take over the world? [/INST]  It is not possible for any individual, including Joe Rogan, to take over the world. Joe Rogan is a popular comedian, mixed martial arts (MMA) commentator, and podcast host, but he does not have any superhuman abilities or special powers that would allow him to control the entire world.

Additionally, the concept of a single individual taking over the world is not a feasible or realistic scenario. The world is a complex and diverse place, with many different countries, cultures, and systems of governance. It would be impossible for any one person to gain control over all of these different entities and maintain their power indefinitely.

In addition, the idea of a single individual taking over the world is often seen as a dangerous and undesirable scenario, as it can lead to authoritarianism, oppression


In [17]:
from huggingface_hub import login
login(token=hf_token)

# push adapters + tokenizer to huggingface
trainer.model.push_to_hub("debsubhra/Llama-2-7b-chat-hf-finetune-test")
tokenizer.push_to_hub("debsubhra/Llama-2-7b-chat-hf-finetune-test")

print("pushed to hf hub")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  560kB /  134MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...phdx8o8g_/tokenizer.model: 100%|##########|  500kB /  500kB            

pushed to hf hub
